# HSLS:09 Dropout Prediction — Modelling Functions

This notebook defines all reusable functions used across the modelling pipeline.
It is imported by the tier-specific modelling notebooks rather than run directly.
All functions assume the clean dataset (hsls_clean.csv), the train/test split
(train_test_split.json), and the feature tier lists (feature_tiers.json) have
already been prepared by the earlier notebooks.

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import json
import shap
from xgboost import XGBClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import classification_report, roc_auc_score, accuracy_score

In [3]:
def fit_models(features, df, train_idx, test_idx, target='X4EVERDROP', refit='f1'):
    
    train = df.loc[train_idx]
    test  = df.loc[test_idx]
    
    features = [f for f in features if f in df.columns]
    
    X_train = train[features]
    y_train = train[target]
    X_test  = test[features]
    y_test  = test[target]
    
    scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()
    
    xgb = XGBClassifier(
        scale_pos_weight=scale_pos_weight,
        eval_metric='logloss',
        random_state=42
    )
    
    param_grid = {
        'n_estimators': [100, 200, 300, 500, 750, 1000],
        'max_depth': [3, 4, 5, 6, 7, 8, 10, 15],
        'learning_rate': [0.005, 0.01, 0.05, 0.1, 0.2],
        'subsample': [0.6, 0.7, 0.8, 0.9, 1.0],
        'colsample_bytree': [0.5, 0.6, 0.7, 0.8, 0.9, 1.0],
    }
    
    scoring = {
        'f1': 'f1',
        'precision': 'precision',
        'recall': 'recall',
        'roc_auc': 'roc_auc'
    }
    
    search = RandomizedSearchCV(
        estimator=xgb,
        param_distributions=param_grid,
        n_iter=100,
        scoring=scoring,
        refit=refit,
        cv=5,
        random_state=42,
        n_jobs=-1,
        verbose=1
    )
    
    search.fit(X_train, y_train)
    
    return search, X_train, y_train, X_test, y_test

In [4]:
def define_fine_grid(cv_results, top_k=3, range_pct=0.2, metric='f1'):
    param_cols = ['param_n_estimators', 'param_max_depth', 'param_learning_rate',
                  'param_subsample', 'param_colsample_bytree']
    
    results = pd.DataFrame(cv_results)
    
    # Filter to competitive region — top range_pct of the score range
    max_score = results[f'mean_test_{metric}'].max()
    min_score = results[f'mean_test_{metric}'].min()
    threshold = max_score - (max_score - min_score) * range_pct
    competitive = results[results[f'mean_test_{metric}'] >= threshold]
    
    top_n_results = competitive.sort_values(f'mean_test_{metric}', ascending=False)[
        [f'mean_test_{metric}', f'std_test_{metric}'] + param_cols
    ]
    
    print(f"Metric: {metric}")
    print(f"Score range: {min_score:.4f} — {max_score:.4f}")
    print(f"Threshold: {threshold:.4f} ({len(top_n_results)} combinations above threshold)")
    print()
    print(top_n_results.to_string())
    print()
    
    for col in param_cols:
        print(top_n_results[col].value_counts())
        print()
    
    fine_grid = {}
    
    for col in param_cols:
        clean_col = col.replace('param_', '')
        counts = top_n_results[col].value_counts().sort_values(ascending=False)
        all_values = sorted(results[col].unique())
        
        kth_count = counts.iloc[min(top_k, len(counts)) - 1]
        selected = sorted(counts[counts >= kth_count].index.tolist())
        
        filled = list(selected)
        for i in range(len(selected) - 1):
            lo = selected[i]
            hi = selected[i + 1]
            between = [v for v in all_values if lo < v < hi]
            filled.extend(between)
        filled = sorted(set(filled))
        
        fine_grid[clean_col] = filled
        print(f"{clean_col}: {filled}")
    
    n_combinations = 1
    for values in fine_grid.values():
        n_combinations *= len(values)
    
    print(f"\nTotal combinations: {n_combinations}")
    print(f"Total fits (x5 folds): {n_combinations * 5}")
    
    return fine_grid

In [5]:
def fit_fine_search(fine_grid, X_train, y_train, refit='f1'):
    
    scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()
    
    xgb = XGBClassifier(
        scale_pos_weight=scale_pos_weight,
        eval_metric='logloss',
        random_state=42
    )
    
    scoring = {
        'f1': 'f1',
        'precision': 'precision',
        'recall': 'recall',
        'roc_auc': 'roc_auc'
    }
    
    grid_search = GridSearchCV(
        estimator=xgb,
        param_grid=fine_grid,
        scoring=scoring,
        refit=refit,
        cv=5,
        n_jobs=-1,
        verbose=1
    )
    
    grid_search.fit(X_train, y_train)
    
    results = pd.DataFrame(grid_search.cv_results_)
    param_cols = [col for col in results.columns if col.startswith('param_')]
    top10 = results.sort_values(f'mean_test_{refit}', ascending=False)[
        ['mean_test_f1', 'std_test_f1',
         'mean_test_precision', 'mean_test_recall',
         'mean_test_roc_auc'] + param_cols
    ].head(10)
    
    print(top10.to_string())
    
    return grid_search

In [6]:
def evaluate_model(fine_search, X_train, y_train, X_test, y_test, tier):
    
    best_model = fine_search.best_estimator_
    
    y_pred_train = best_model.predict(X_train)
    y_prob_train = best_model.predict_proba(X_train)[:, 1]
    
    print(f"Tier {tier} — Training Set Evaluation")
    print(f"Accuracy:  {accuracy_score(y_train, y_pred_train):.4f}")
    print(f"ROC-AUC:   {roc_auc_score(y_train, y_prob_train):.4f}")
    print()
    print(classification_report(y_train, y_pred_train))
    
    y_pred = best_model.predict(X_test)
    y_prob = best_model.predict_proba(X_test)[:, 1]
    
    print(f"Tier {tier} — Test Set Evaluation")
    print(f"Accuracy:  {accuracy_score(y_test, y_pred):.4f}")
    print(f"ROC-AUC:   {roc_auc_score(y_test, y_prob):.4f}")
    print()
    print(classification_report(y_test, y_pred))

In [7]:
def show_feature_importance(fine_search, X_train, tier, top_n=20):
    model = fine_search.best_estimator_
    importance = pd.Series(model.feature_importances_,
                           index=X_train.columns)
    importance = importance.sort_values(ascending=False)
    
    # Plot top 20
    top = importance.head(top_n)
    fig, ax = plt.subplots(figsize=(10, 8))
    top.sort_values().plot(kind='barh', ax=ax)
    ax.set_title(f'Tier {tier} — Top {top_n} Feature Importances')
    ax.set_xlabel('Importance')
    plt.tight_layout()
    plt.show()
    
    # Print bottom 20
    print(f"Tier {tier} — Bottom {top_n} features:")
    print(importance.tail(top_n).to_string())
    print()

In [8]:
def compare_feature_importance(fine_searches, X_trains, top_n=20):
    
    # Get top 20 features for each tier
    tier_tops = {}
    for tier, (fine_search, X_train) in enumerate(zip(fine_searches, X_trains), start=1):
        model = fine_search.best_estimator_
        importance = pd.Series(model.feature_importances_, index=X_train.columns)
        importance = importance.sort_values(ascending=False)
        tier_tops[tier] = importance.head(top_n).index.tolist()
    
    # Find overlap — features appearing in top 20 of multiple tiers
    all_top_features = set()
    for features in tier_tops.values():
        all_top_features.update(features)
    
    print("Feature overlap across tiers (top 20 each):")
    print(f"{'Feature':<25} {'T1':>4} {'T2':>4} {'T3':>4} {'T4':>4} {'Tiers':>6}")
    print("-" * 50)
    
    overlap = []
    for feature in all_top_features:
        ranks = []
        for tier, features in tier_tops.items():
            if feature in features:
                ranks.append((tier, features.index(feature) + 1))
        if len(ranks) > 1:
            overlap.append((feature, ranks))
    
    overlap.sort(key=lambda x: -len(x[1]))
    
    for feature, ranks in overlap:
        rank_dict = dict(ranks)
        t1 = str(rank_dict.get(1, '-'))
        t2 = str(rank_dict.get(2, '-'))
        t3 = str(rank_dict.get(3, '-'))
        t4 = str(rank_dict.get(4, '-'))
        n_tiers = len(ranks)
        print(f"{feature:<25} {t1:>4} {t2:>4} {t3:>4} {t4:>4} {n_tiers:>6}")
    
    # Average rank across tiers where feature appears in top 20
    print("\nAverage rank across tiers (only where in top 20):")
    print(f"{'Feature':<25} {'Avg Rank':>10} {'Tiers':>6}")
    print("-" * 45)
    
    avg_ranks = []
    for feature, ranks in overlap:
        avg_rank = sum(r for _, r in ranks) / len(ranks)
        avg_ranks.append((feature, avg_rank, len(ranks)))
    
    avg_ranks.sort(key=lambda x: (-x[2], x[1]))
    
    for feature, avg_rank, n_tiers in avg_ranks:
        print(f"{feature:<25} {avg_rank:>10.2f} {n_tiers:>6}")

In [9]:
def plot_shap(fine_search, X_train, tier, max_display=20):
    model = fine_search.best_estimator_
    explainer = shap.TreeExplainer(model)
    shap_values = explainer.shap_values(X_train)
    
    print(f"Tier {tier} — SHAP Summary Plot")
    shap.summary_plot(shap_values, X_train, max_display=max_display, show=True)

In [10]:
def ensemble_predictions(fine_searches, X_tests, y_test, thresholds=[1,2,3,4]):
    
    # Get predicted probabilities from each tier model
    probs = []
    for fine_search, X_test in zip(fine_searches, X_tests):
        model = fine_search.best_estimator_
        prob = model.predict_proba(X_test)[:, 1]
        probs.append(prob)
    
    probs_df = pd.DataFrame({
        'tier1': probs[0],
        'tier2': probs[1],
        'tier3': probs[2],
        'tier4': probs[3],
        'true_label': y_test.values
    })
    
    # Average probability across all four models
    probs_df['avg_prob'] = probs_df[['tier1','tier2','tier3','tier4']].mean(axis=1)
    
    # Hard vote — how many models predict dropout at 0.5 threshold
    for col in ['tier1','tier2','tier3','tier4']:
        probs_df[f'{col}_pred'] = (probs_df[col] >= 0.5).astype(int)
    
    probs_df['votes'] = probs_df[['tier1_pred','tier2_pred',
                                   'tier3_pred','tier4_pred']].sum(axis=1)
    
    print("=== Probability Averaging Ensemble ===")
    for threshold in [0.3, 0.4, 0.5, 0.6]:
        y_pred = (probs_df['avg_prob'] >= threshold).astype(int)
        print(f"\nThreshold {threshold}:")
        print(classification_report(probs_df['true_label'], y_pred))
    
    print("\n=== Vote-Based Ensemble ===")
    for min_votes in [1, 2, 3, 4]:
        y_pred = (probs_df['votes'] >= min_votes).astype(int)
        p = precision_score(probs_df['true_label'], y_pred)
        r = recall_score(probs_df['true_label'], y_pred)
        f = f1_score(probs_df['true_label'], y_pred)
        print(f"Min votes={min_votes}: Precision={p:.3f}, Recall={r:.3f}, F1={f:.3f}")
    
    return probs_df